# Notebook de Pruebas — Sistema Pre-ICFES
Prueba cada componente por separado antes de usarlo en la app.

**Corre las celdas en orden. Cada una es independiente y muestra el resultado paso a paso.**

In [ ]:
# ══ PASO 0 — Instalar dependencias ══
!pip install gspread opencv-python-headless pytesseract easyocr --quiet
!apt-get install -y tesseract-ocr tesseract-ocr-spa -qq
print("Listo")


In [ ]:
# ══ PASO 1 — Conectar al Sheets ══
# Verifica que puedes leer las pestañas

import gspread
from google.colab import auth
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEETS_ID = "1cdKlRv6vh8UJxXAPtAWlpRsx5DVuBqZrMryAn5FVriM"
sh = gc.open_by_key(SHEETS_ID)

print(f'Sheets conectado: {sh.title}')
print()
pestanas = ['dim_colegios','dim_grupos','dim_estudiantes','dim_pruebas','dim_item','dim_respuestas','fact_calificacion']
for p in pestanas:
    try:
        ws = sh.worksheet(p)
        n  = len(ws.get_all_values()) - 1
        print(f'  ✅  {p:<25} {n} filas')
    except:
        print(f'  ❌  {p:<25} NO ENCONTRADA')


In [ ]:
# ══ PASO 2 — Leer colegios y pruebas activas ══
# Lo que verá el monitor en los primeros desplegables

colegios = sh.worksheet('dim_colegios').get_all_records()
activos  = [c for c in colegios if str(c.get('activo','')).upper() == 'SI']
print(f'Colegios activos ({len(activos)}):')
for c in activos:
    print(f"  colegio_id='{c['colegio_id']}' | nombre='{c['nombre']}'")

print()
pruebas = sh.worksheet('dim_pruebas').get_all_records()
activas = [p for p in pruebas if str(p.get('activa','')).upper() == 'SI']
print(f'Pruebas activas ({len(activas)}):')
for p in activas:
    print(f"  prueba_id='{p['prueba_id']}' | nombre='{p['nombre_display']}'")


In [ ]:
# ══ PASO 3 — Grupos de un colegio ══
# Cambia COLEGIO_TEST por el colegio que quieras probar

COLEGIO_TEST = 'CISF'  # ← cambia aquí

grupos = sh.worksheet('dim_grupos').get_all_records()
grupos_col = [g for g in grupos if str(g.get('colegio_id','')) == COLEGIO_TEST]
print(f'Grupos de {COLEGIO_TEST} ({len(grupos_col)}):')
for g in grupos_col:
    print(f"  grupo_id='{g['grupo_id']}' | grado={g['grado']} | nombre='{g['nombre']}'")


In [ ]:
# ══ PASO 4 — Estudiantes de un grupo ══
# Cambia GRUPO_TEST por el grupo que quieras probar

GRUPO_TEST = 'CISF-3A-2026'  # ← cambia aquí

ests = sh.worksheet('dim_estudiantes').get_all_records()
ests_grupo = [e for e in ests if str(e.get('grupo_id','')) == GRUPO_TEST]
print(f'Estudiantes en {GRUPO_TEST} ({len(ests_grupo)}):')
for e in ests_grupo[:10]:
    print(f"  id='{e['estudiante_id']}' | {e['nombres']} {e['apellidos']}")
if len(ests_grupo) > 10:
    print(f'  ... y {len(ests_grupo)-10} más')
if not ests_grupo:
    print('  ⚠️  Sin estudiantes. ¿El grupo_id es correcto?')


In [ ]:
# ══ PASO 5 — Probar normalización de curso ══
# Simula lo que el OCR puede leer y cómo se convierte

import re

NUMEROS = {
    "primero":"1","segundo":"2","tercero":"3","cuarto":"4",
    "quinto":"5","sexto":"6","septimo":"7","séptimo":"7",
    "octavo":"8","noveno":"9","decimo":"10","décimo":"10",
    "once":"11","undecimo":"11","undécimo":"11",
}

def normalizar_curso(raw):
    if not raw or not raw.strip(): return None
    s = raw.strip().lower()
    s = re.sub(r"[°ºo\.\-_]", "", s)
    letras = re.findall(r"\b([a-e])\b", s)
    letra  = letras[-1].upper() if letras else ""
    numero = None
    for palabra, num in NUMEROS.items():
        if palabra in s: numero = num; break
    if numero is None:
        nums = re.findall(r"\b(1[01]|[1-9])\b", s)
        if nums: numero = nums[0]
    return f"{numero}{letra or 'A'}" if numero else None

casos_prueba = [
    "3A", "3a", "3 A", "Tercero A", "tercero a",
    "3", "Tercero", "3B", "Primero", "1",
    "11A", "Once A", "10B", "Décimo B",
    "", None, "abc"
]

print('Entrada               → Normalizado')
print('-'*40)
for caso in casos_prueba:
    resultado = normalizar_curso(caso) if caso is not None else None
    print(f"  {str(caso):<20} → {resultado or '❌ no reconocido'}")


In [ ]:
# ══ PASO 6 — OCR con Gemini Vision (recomendado) ══
# Gemini lee texto impreso + manuscrito en formularios sin preprocesamiento
#
# Obtén tu API key GRATIS en: https://aistudio.google.com/apikey

GEMINI_API_KEY = ''  # ← pega tu key aquí

# ─────────────────────────────────────────────────────────────────────────
import subprocess, sys
try:
    import google.generativeai as genai
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','google-generativeai','--quiet'], check=True)
    import google.generativeai as genai

from google.colab import files
import cv2, numpy as np, re, io, json as _json
import matplotlib.pyplot as plt
from PIL import Image

PROMPT_EXTRACCION = '''
Analiza este encabezado de una hoja de respuestas de un simulacro escolar colombiano.
Hay campos impresos con valores escritos a mano por el estudiante.

Extrae exactamente estos campos y responde ÚNICAMENTE con JSON válido (sin markdown, sin explicación):
{
  "nombres": "nombre(s) del estudiante",
  "apellidos": "apellido(s) del estudiante",
  "curso": "curso o grado (ej: 1, 3A, 10B)",
  "num_doc": "número de documento solo dígitos",
  "docente": "nombre del docente",
  "fecha": "fecha en formato DD-MM-AAAA"
}

Si un campo no es legible usa null. El campo num_doc debe contener SOLO dígitos.
'''

def recortar_encabezado(img_bytes, fraccion=0.28):
    nparr  = np.frombuffer(img_bytes, np.uint8)
    img_cv = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    h, w   = img_cv.shape[:2]
    if max(h,w) < 1800:
        f = 1800/max(h,w)
        img_cv = cv2.resize(img_cv,(int(w*f),int(h*f)),interpolation=cv2.INTER_CUBIC)
        h, w = img_cv.shape[:2]
    return img_cv[0:int(h*fraccion), 0:w]

def extraer_con_gemini(img_bgr, api_key):
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-2.0-flash')
    # Convertir a JPEG bytes
    ok, buf = cv2.imencode('.jpg', img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 92])
    img_bytes_jpg = buf.tobytes()
    response = model.generate_content([
        {'mime_type': 'image/jpeg', 'data': img_bytes_jpg},
        PROMPT_EXTRACCION
    ])
    texto = response.text.strip()
    # Limpiar posible markdown
    texto = re.sub(r'^```[a-z]*\n?','', texto)
    texto = re.sub(r'\n?```$','', texto).strip()
    return _json.loads(texto)

# ─────────────────────────────────────────────────────────────────────────
print('Sube una foto de la hoja de respuestas...')
uploaded = files.upload()

if uploaded:
    nombre    = list(uploaded.keys())[0]
    img_bytes = uploaded[nombre]

    img_pil = Image.open(io.BytesIO(img_bytes))
    plt.figure(figsize=(6,8))
    plt.imshow(img_pil); plt.title('Foto subida'); plt.axis('off'); plt.show()

    enc = recortar_encabezado(img_bytes)
    plt.figure(figsize=(12,3))
    plt.imshow(cv2.cvtColor(enc, cv2.COLOR_BGR2RGB))
    plt.title('Zona del encabezado enviada a Gemini'); plt.axis('off'); plt.show()

    if not GEMINI_API_KEY:
        print('⚠️  Pon tu API key en GEMINI_API_KEY (gratis en aistudio.google.com/apikey)')
    else:
        print('Enviando a Gemini Vision...')
        try:
            campos_ocr = extraer_con_gemini(enc, GEMINI_API_KEY)
            # Limpiar num_doc
            if campos_ocr.get('num_doc'):
                campos_ocr['num_doc'] = re.sub(r'[^0-9]','', str(campos_ocr['num_doc']))
            # Normalizar curso
            campos_ocr['curso_norm'] = normalizar_curso(str(campos_ocr.get('curso') or ''))

            print('\n' + '='*45)
            print('CAMPOS EXTRAÍDOS POR GEMINI')
            print('='*45)
            for k, v in campos_ocr.items():
                print(f'  {"✅" if v else "⚠️ "} {k:<15} {v or "(no leido)"}')
            print('='*45)

            # Variable texto_raw para compatibilidad con PASO 7
            texto_raw = (
                f"Nombre: {campos_ocr.get('nombres','')}\n"
                f"Apellido: {campos_ocr.get('apellidos','')}\n"
                f"Curso: {campos_ocr.get('curso','')}\n"
                f"N de Documento: {campos_ocr.get('num_doc','')}\n"
                f"Docente: {campos_ocr.get('docente','')}\n"
                f"Fecha: {campos_ocr.get('fecha','')}\n"
            )
        except Exception as e:
            print(f'❌ Error: {e}')
            print('Verifica que la API key sea válida y que el modelo gemini-2.0-flash esté disponible.')


In [ ]:
# ══ PASO 7 — Parsear campos del texto OCR ══
# Ejecutar después del PASO 6

def parsear_texto(texto):
    campos = {k:None for k in ['nombres','apellidos','num_doc','curso_raw','curso_norm','docente','fecha']}
    txt = ' '.join(l.strip() for l in texto.split('\n') if l.strip()).lower()
    print(f'Texto unificado:\n  {txt[:200]}\n')

    def buscar(patron):
        m = re.search(
            rf'{patron}[:\s]+([A-Za-záéíóúñÁÉÍÓÚÑ0-9\s\-\.]+?)'
            r'(?=\s*(?:nombre|apellido|documento|docente|curso|fecha|n[°º]|$))',
            txt, re.IGNORECASE)
        if m:
            v = m.group(1).strip().rstrip('_-').strip()
            return v.title() if len(v) > 1 else None
        return None

    campos['nombres']   = buscar(r'nombre')
    campos['apellidos'] = buscar(r'apellido')
    m = re.search(r'(?:documento|n[°º]\s*doc)[:\s]+([0-9]{5,12})', txt)
    if m: campos['num_doc'] = m.group(1).strip()
    m = re.search(r'curso[:\s]+([A-Za-záéíóúñ0-9°º\s\-\.]+?)(?=\s*(?:fecha|docente|$))', txt)
    if m:
        campos['curso_raw']  = m.group(1).strip()
        campos['curso_norm'] = normalizar_curso(campos['curso_raw'])
    return campos

campos = parsear_texto(texto_raw)

print('Campos extraídos:')
print('-'*40)
for k, v in campos.items():
    estado = '✅' if v else '⚠️  N/A'
    print(f'  {k:<15} {estado}  {v or ""}')

# Buscar grupo sugerido
COLEGIO_TEST_OCR = 'CISF'  # ← cambia al colegio de la prueba
if campos['curso_norm']:
    candidato = f"{COLEGIO_TEST_OCR}-{campos['curso_norm']}-2026"
    grupos    = sh.worksheet('dim_grupos').get_all_records()
    match     = next((g for g in grupos if g.get('grupo_id','') == candidato), None)
    print(f'\nGrupo sugerido: {candidato}')
    print(f'  Existe en dim_grupos: {"✅ SÍ" if match else "❌ NO"}')
else:
    print('\n⚠️  No se pudo determinar el curso — el monitor lo seleccionará manualmente')


In [ ]:
# ══ PASO 8 — Probar OMR (detección de respuestas) ══
# Usa la misma foto del PASO 6

from scipy.ndimage import uniform_filter1d

def procesar_omr(imagen_bytes):
    nparr  = np.frombuffer(imagen_bytes, np.uint8)
    imagen = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    h, w   = imagen.shape[:2]
    if max(h,w) > 2500:
        f = 2500/max(h,w); imagen = cv2.resize(imagen,(int(w*f),int(h*f)))
    ratio = h/w
    if ratio > 1.2:
        zt,zb,zl,zr = 0.21,0.78,0.08,0.98
        if max(h,w)<1500: f=1500/max(h,w); imagen=cv2.resize(imagen,(int(w*f),int(h*f))); h,w=imagen.shape[:2]
    else:
        zt,zb,zl,zr = 0.32,0.96,0.15,1.00
        if max(h,w)<1000: f=1400/max(h,w); imagen=cv2.resize(imagen,(int(w*f),int(h*f))); h,w=imagen.shape[:2]
    hoja = imagen
    h2,w2 = hoja.shape[:2]
    zona  = hoja[int(h2*zt):int(h2*zb), int(w2*zl):int(w2*zr)]
    gris  = cv2.cvtColor(zona, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gris,(5,5),0)
    thresh= cv2.adaptiveThreshold(blur,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,11,2)
    cnts,_ = cv2.findContours(thresh,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    burbujas=[]
    for c in cnts:
        area=cv2.contourArea(c)
        if area<80: continue
        x,y,bw,bh=cv2.boundingRect(c)
        if not(12<=bw<=90 and 12<=bh<=90): continue
        if not(0.60<=bw/float(bh)<=1.60): continue
        perim=cv2.arcLength(c,True)
        if perim==0: continue
        if 4*np.pi*area/(perim**2)<0.45: continue
        cx=x+bw//2; cy=y+bh//2
        r_inner=max(max(bw,bh)//2-4,3)
        mask=np.zeros(gris.shape,dtype=np.uint8)
        cv2.circle(mask,(cx,cy),r_inner,255,-1)
        if cv2.countNonZero(mask)==0: continue
        L=cv2.mean(gris,mask=mask)[0]
        burbujas.append({'cx':cx,'cy':cy,'r':max(bw,bh)//2,'L':L})
    return burbujas, zona

burbujas, zona = procesar_omr(img_bytes)
print(f'Burbujas detectadas: {len(burbujas)}')

# Mostrar zona con burbujas marcadas
vis = zona.copy()
for b in burbujas:
    cv2.circle(vis,(b['cx'],b['cy']),b['r'],(0,255,0),2)
plt.figure(figsize=(12,8))
plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB))
plt.title(f'Burbujas detectadas: {len(burbujas)}')
plt.axis('off')
plt.show()


In [ ]:
# ══ PASO 9 — Probar match con dim_item ══
# Simula el calculo de puntaje para un estudiante

PRUEBA_TEST     = 'PRESABER-1-2026-V1'  # ← cambia aquí
ESTUDIANTE_TEST = 'CISF-EST-12345'       # ← cambia aquí

# Leer items de la prueba
items = sh.worksheet('dim_item').get_all_records()
items_prueba = [i for i in items if str(i.get('prueba_id','')) == PRUEBA_TEST]
print(f'Items en {PRUEBA_TEST}: {len(items_prueba)}')

if items_prueba:
    print('\nPrimeros 5 items:')
    for it in sorted(items_prueba, key=lambda x: str(x.get('num_pregunta','')))[:5]:
        print(f"  P{it.get('num_pregunta','?')} | area={it.get('area_id')} | clave={it.get('clave_correcta')} | peso={it.get('peso')}")

# Simular respuestas aleatorias
import random
respuestas_sim = {i+1: random.choice(['A','B','C','D']) for i in range(len(items_prueba))}

# Calcular puntaje
correctas = 0; puntaje = 0; puntaje_max = 0
por_area  = {}
for item in sorted(items_prueba, key=lambda x: str(x.get('num_pregunta',''))):
    digits   = ''.join(c for c in str(item.get('num_pregunta','')) if c.isdigit())
    num_int  = int(digits) if digits else 0
    resp     = respuestas_sim.get(num_int, '')
    clave    = str(item.get('clave_correcta',''))
    es_corr  = 1 if resp == clave else 0
    peso     = float(item.get('peso', 0))
    area     = str(item.get('area_id',''))
    correctas   += es_corr
    puntaje     += peso * es_corr
    puntaje_max += peso
    if area not in por_area: por_area[area] = {'c':0,'t':0}
    por_area[area]['t'] += 1
    por_area[area]['c'] += es_corr

total = len(items_prueba)
pct   = round(puntaje/puntaje_max*100,1) if puntaje_max > 0 else 0
print(f'\nRESULTADO SIMULADO:')
print(f'  Correctas  : {correctas}/{total}')
print(f'  Puntaje    : {round(puntaje,1)}/{round(puntaje_max,1)}')
print(f'  Porcentaje : {pct}%')
print(f'\nPor área:')
for area, v in por_area.items():
    p2 = round(v['c']/v['t']*100,1) if v['t']>0 else 0
    print(f'  {area}: {v["c"]}/{v["t"]} ({p2}%)')


In [ ]:
# ══ PASO 10 — Verificar escritura en Sheets ══
# Escribe UNA FILA DE PRUEBA en dim_respuestas y la borra después

from datetime import datetime, timezone, timedelta

tz_col = timezone(timedelta(hours=-5))
ts     = datetime.now(tz_col).strftime('%Y-%m-%d %H:%M:%S')

# Fila de prueba
fila_test = ['TEST-EST-0000','PRESABER-1-2026-V1','TEST-GRUPO','TEST-COL',ts]
for i in range(100): fila_test.append('A')

ws_resp = sh.worksheet('dim_respuestas')
ws_resp.append_row(fila_test, value_input_option='RAW')
print('✅ Fila de prueba escrita en dim_respuestas')

# Contar filas
todas = ws_resp.get_all_values()
print(f'  Total filas ahora: {len(todas)-1} (sin encabezado)')

# Borrar la fila de prueba
ws_resp.delete_rows(len(todas))
print('✅ Fila de prueba borrada — Sheets intacto')

print()
print('Todo funciona correctamente. Listo para usar en la app.')
